# Project Part 3: Base Map

## Overview

In Part 2 you produced a cleaned, sentiment-annotated dataset and a written analysis. Now you will build the **base map** that will power the interactive visualization on your team website.

This map is the foundation for the flythrough you will build in **Part 4**. It needs to:

- Display all geoparsed locations for your school, sized by the number of times it occurs and colored by sentiment
- Be clean enough to serve as a standalone visualization
- Export in a format usable by the flythrough template

---

## ⚠️ Before You Begin

You must have completed **[project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb)** and created a cleaned data set of your school's data. You should see a copy of it in your school's data folder called `{SCHOOL}_geoparsed_long_cleaned_sentiment.csv`, where `{SCHOOL}` is your school's abbreviation. If along the way you find that the data was not properly cleaned, you have to repeat steps 2–5 in **[project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb)**.

In [1]:
# ============================================================
# STEP 0: Set your school (must match Part 2)
# ============================================================

SCHOOL = "UNC"   # <-- change this to your school

import pandas as pd
import plotly.express as px
import json

---

## 📖 1 Follow Along — Load and Aggregate Your Processed Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The cell below loads sentiment data for **both your school and JMU**, then aggregates each dataset to one row per unique place. The two datasets are combined into a single DataFrame before any classification. This mirrors the approach from [Part 2](project_part_2_whitepaper.ipynb) and ensures that the Jenks size and color bins computed in Section 2 are shared across both schools — so equivalent bubbles carry equivalent meaning on the map.

The `school` column records which school each place belongs to; it will appear in the hover tooltip.

If your school's file is not found, complete `project_part_1_data_pipeline.ipynb` for your school first.

In [2]:
import os
import numpy as np

SCHOOL_DATA_PATH = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned_sentiment.pickle'
JMU_PICKLE_PATH  = '../data/JMU/JMU_geoparsed_long_cleaned_sentiment.pickle'
JMU_CSV_PATH     = '../data/JMU/JMU_geoparsed_long_cleaned_sentiment.csv'
JMU_BACKUP_PATH  = '../data/JMU/JMU_geoparsed_long_backup_sentiment.csv'

df_places = None

def _aggregate(df_raw, school_name):
    """Aggregate a per-sentence DataFrame to one row per place."""
    return (
        df_raw
        .dropna(subset=['place', 'latitude', 'longitude'])
        .astype({'latitude': float, 'longitude': float})
        .groupby('place', sort=False)
        .agg(
            location_count=('place', 'size'),
            latitude=('latitude', 'first'),
            longitude=('longitude', 'first'),
            sentences=('sentences', lambda x: ' | '.join(str(s) for s in list(x)[:5])),
            avg_roberta_compound=('roberta_compound', 'mean'),
            place_type=('place_type', 'first'),
        )
        .reset_index()
        .assign(school=school_name)
    )

# ── Load school data ──────────────────────────────────────────────────────────
if not os.path.exists(SCHOOL_DATA_PATH):
    print(f'⛔ File not found: {SCHOOL_DATA_PATH}')
    print('   Complete project_part_1_data_pipeline.ipynb for your school first.')
else:
    df_school_raw = pd.read_pickle(SCHOOL_DATA_PATH)
    print(f'✅ {SCHOOL}: loaded {len(df_school_raw):,} rows')

    # ── Load JMU data (pickle → primary CSV → backup CSV) ────────────────────────
    if os.path.exists(JMU_PICKLE_PATH):
        df_jmu_raw = pd.read_pickle(JMU_PICKLE_PATH)
    elif os.path.exists(JMU_CSV_PATH):
        df_jmu_raw = pd.read_csv(JMU_CSV_PATH)
        print('⚠️  Using JMU primary CSV (pickle not found).')
    elif os.path.exists(JMU_BACKUP_PATH):
        df_jmu_raw = pd.read_csv(JMU_BACKUP_PATH)
        print('⚠️  Using JMU backup sentiment data.')
    else:
        raise FileNotFoundError('No JMU sentiment data found in ../data/JMU/.')
    print(f'✅ JMU:  loaded {len(df_jmu_raw):,} rows')

    # ── Aggregate each school to one row per place ────────────────────────────────
    df_school = _aggregate(df_school_raw, SCHOOL)
    df_jmu    = _aggregate(df_jmu_raw,    'JMU')

    # ── Combine — Jenks bins in Section 2 will be shared across both schools ──────
    df_places = pd.concat([df_jmu, df_school], ignore_index=True)

    print(f'\n✅ JMU: {len(df_jmu):,} unique places  |  {SCHOOL}: {len(df_school):,} unique places')
    print(f'✅ Combined: {len(df_places):,} total place records')
    print(f'\nTop place types (combined):')
    print(df_places['place_type'].value_counts(dropna=False).head(8).to_string())
    print(f'\nSentiment range: {df_places["avg_roberta_compound"].min():.3f} to {df_places["avg_roberta_compound"].max():.3f}')
    df_places.head(3)

✅ UNC: loaded 1,263 rows
⚠️  Using JMU backup sentiment data.
✅ JMU:  loaded 876 rows

✅ JMU: 307 unique places  |  UNC: 369 unique places
✅ Combined: 676 total place records

Top place types (combined):
place_type
City               204
Building           156
State              103
Country             56
Natural Feature     41
None                40
Neighborhood        28
Region              20

Sentiment range: -0.949 to 0.984


---

## 2 Build Your Map

Fill in the design brief table below **before** changing any code. Every parameter should be a deliberate choice — not an accepted default. Reference your observations from [Lesson 6](../lesson_6_mapping_fundamentals/lesson_6_mapping_fundamentals.ipynb) Decisions 1–7.

> ✍️ **Activity:** Complete every row in the design brief, then set each matching variable in the code cell and run it to generate your map.

### Design Brief

**Fill in every row before touching the code cell.** Reference Decisions 1–7 from Lesson 6.

| Design Decision | Variable | Your Choice | Reasoning |
|---|---|---|---|
| Filtering threshold | `MIN_COUNT` | | Which places are worth showing? |
| Place type filter | `PLACE_TYPES` | | Which geographic scales belong in your story? |
| Size classification | `N_SIZE_CLASSES` | | How many Jenks size classes? (3–5) |
| Color buckets | `N_COLOR_BUCKETS` | | How many sentiment color classes? (3–7) |
| Color scale | `COLOR_SCALE` | | Which scale is honest and accessible? |
| Maximum bubble size | `SIZE_MAX` | | How large should the biggest bubble be in pixels? |
| Base map style | `MAP_STYLE` | | What tone does the background set? |
| Center coordinates | `CENTER` | | Reference viewport — flythrough will override |
| Zoom level | `ZOOM` | | Reference zoom — flythrough will override |

> 👉 **Note:** *Your submitted map must differ from the default values in at least three deliberate ways, each justifiable from this brief.*

In [6]:
import mapclassify
import plotly.colors as pc

if df_places is None:
    print('⛔ No data — run the cells in Section 1 first.')
else:
    # ── Your design decisions — fill in every value, then run ────────────────────
    MIN_COUNT       = 3              # ← Decision 1: minimum post count per location
    PLACE_TYPES     = None           # ← Decision 2: list e.g. ['City', 'Building'], or None for all types
    N_SIZE_CLASSES  = 4              # ← Decision 3: number of Jenks size classes (try 3–5)
    N_COLOR_BUCKETS = 5              # ← Decision 4: number of sentiment color buckets (try 3–7)
    COLOR_SCALE     = 'RdYlGn'      # ← Decision 5: 'RdYlGn', 'RdBu', 'Spectral', 'Viridis'
    SIZE_MAX        = 18             # ← maximum bubble diameter in pixels (try 12–30)
    MAP_STYLE       = 'carto-positron'  # ← Decision 6: 'carto-positron', 'carto-darkmatter', 'open-street-map'
    CENTER          = {"lat": 37.5, "lon": -78.0}  # ← reference only; flythrough will override this
    ZOOM            = 6              # ← reference only; flythrough will override this

    # ── Filter ────────────────────────────────────────────────────────────────────
    if PLACE_TYPES is not None:
        df_work = df_places[
            (df_places['location_count'] >= MIN_COUNT) &
            df_places['place_type'].isin(PLACE_TYPES)
        ].copy()
    else:
        df_work = df_places[df_places['location_count'] >= MIN_COUNT].copy()

    _n_jmu = (df_work['school'] == 'JMU').sum()
    _n_sch = (df_work['school'] == SCHOOL).sum()
    print(f'── After filtering: JMU {_n_jmu}, {SCHOOL} {_n_sch}  (total {len(df_work)}) ──')

    # ── Size classification — Jenks on combined data ──────────────────────────────
    _jnb_s = mapclassify.NaturalBreaks(df_work['location_count'].values, k=N_SIZE_CLASSES)
    df_work['size_class'] = (_jnb_s.yb + 1).astype(float)

    # ── Color classification — Jenks on combined data ─────────────────────────────
    scores  = df_work['avg_roberta_compound']
    _jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
    _breaks = _jnb_c.bins
    _lo     = scores.min()
    _labels = []
    for _hi in _breaks:
        _labels.append(f"{_lo:.2f} to {_hi:.2f}")
        _lo = _hi
    df_work['color_class'] = pd.cut(scores, bins=[-float('inf')] + list(_breaks), labels=_labels)
    _palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
    _color_map = dict(zip(_labels, _palette))

    # ── Build the map ─────────────────────────────────────────────────────────────
    fig = px.scatter_map(
        df_work,
        lat='latitude', lon='longitude',
        size='size_class',
        color='color_class',
        hover_name='place',
        hover_data={
            'school': True,
            'avg_roberta_compound': ':.3f',
            'location_count': True,
            'place_type': True,
            'size_class': False,
            'color_class': False,
            'latitude': False,
            'longitude': False,
        },
        color_discrete_map=_color_map,
        category_orders={'color_class': _labels},
        size_max=SIZE_MAX,
        map_style=MAP_STYLE,
        center=CENTER,
        zoom=ZOOM,
        height=650,
        title=f'JMU & {SCHOOL} — Base Map  ({len(df_work):,} locations)',
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
    fig.show()

── After filtering: JMU 55, UNC 73  (total 128) ──


---

## ✍️ 3 Prepare Locations for the Flythrough

The flythrough template reads location data from `flythrough_config.js`. You need to export the top locations from your map — the ones you want to "fly through" — in the format that file expects.

> ✍️ **Activity:** Adjust `N_FLYTHROUGH` and the sort column to select the locations that tell the best story. Run the cells, then copy the JSON output into the `locations` array in `flythrough_config.js`.

In [ ]:
if 'df_work' not in dir() or df_work is None:
    print('⛔ Run the map cell in Section 2 first.')
else:
    # ── Adjust these values ────────────────────────────────────────────────────────
    N_FLYTHROUGH = 10      # ← number of locations to fly through (aim for 5–15)
    SORT_BY      = 'location_count'  # ← 'location_count' (most discussed) or 'avg_roberta_compound' (most extreme sentiment)

    top_locations = (
        df_work
        .sort_values(SORT_BY, ascending=False)
        .head(N_FLYTHROUGH)
        [['place', 'school', 'latitude', 'longitude', 'location_count', 'avg_roberta_compound', 'place_type']]
        .reset_index(drop=True)
    )
    top_locations

In [ ]:
if 'top_locations' not in dir():
    print('⛔ Run the cell above first.')
else:
    # Export to the format expected by flythrough_config.js
    locations_json = []
    for _, row in top_locations.iterrows():
        locations_json.append({
            "name": row['place'],
            "school": row['school'],
            "lat": float(row['latitude']),
            "lon": float(row['longitude']),
            "count": int(row['location_count']),
            "sentiment": round(float(row['avg_roberta_compound']), 3),
        })

    print(json.dumps(locations_json, indent=2))
    print(f'\n✅ Copy the JSON above into the locations array in flythrough_config.js')

---

## Section 4: Embed the Map in the Website

Save the final map as an HTML file so it can be linked from `index.html` or embedded in `whitepaper.html`.

**When your base map is ready, move on to `project_part_4_flythrough.ipynb`.**

In [ ]:
if 'fig' not in dir():
    print('⛔ Run the map cell in Section 2 first.')
else:
    fig.write_html("base_map.html")
    print("✅ Map saved to base_map.html")